## Packages

In [ ]:
library(tidyverse)
library(lme4)
library(lmerTest)
library(broom)
library(broom.mixed)
library(ggeffects)
library(emmeans)
library(performance)
library(patchwork)
library(cowplot)
library(modelsummary)
library(datawizard)
library(see)
# source("scripts/simulate_lmm_data.R")

In [ ]:
simulate_lmm_data <- function() {
 
set.seed(123)
 
n_subjects <- 30
n_time <- 6
 
subject_df <- tibble(
  subject = factor(1:n_subjects),
  b0 = rnorm(n_subjects, mean = 0, sd = 6),
  b1 = rnorm(n_subjects, mean = 0, sd = 1.2)
)
 
df <- expand_grid(
  subject = factor(1:n_subjects),
  time = 0:(n_time - 1)
) |>
  left_join(subject_df, by = "subject") |>
  mutate(
    y = 20 + 2.5 * time + b0 + b1 * time + rnorm(n(), mean = 0, sd = 2)
  )
  return(df)
  }

In [ ]:
list.files( "/sharedscratch/maff1-group/data" )

## Opening SOMA Scan

In [ ]:
# Reading SOMA Scan File
files <- system( "tar -tzf /sharedscratch/maff1-group/data/ADNI_Protein_SOMAscan7k.tar.gz", intern = TRUE )
files

In [ ]:
# Opening Methods PDF
system("tar -xf /sharedscratch/maff1-group/data/ADNI_Protein_SOMAscan7k.tar.gz ADNI_CruchagaLab_Methods_CSF_SomaScan7K_Proteomic_Data_20230817.pdf")

In [ ]:
cmd <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_Protein_SOMAscan7k.tar.gz CruchagaLab_CSF_SOMAscan7k_Protein_matrix_postQC_20230620.csv"
somascan1 <- read.csv( pipe(cmd) )

In [ ]:
cmd <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_Protein_SOMAscan7k.tar.gz ADNI_Cruchaga_lab_CSF_SOMAscan7k_analyte_information_20_06_2023.csv"
somascan2 <- read.csv( pipe(cmd) )

In [ ]:
table(somascan2$Target)
# GSK-3 beta x2
# GSK-3 alpha 
# Insulin
# TNF-a
# IL-6
# IL-1b

In [ ]:
# Finding Codes for GSK-3 beta, GSK-3 alpha, & Insulin
somascan_GSKB <- somascan2 |>
    subset(Target == "GSK-3 beta")
somascan_GSKA <- somascan2 |>
    subset(Target == "GSK-3 alpha")
somascan_INS <- somascan2 |>
    subset(Target == "Insulin")
somscan_TNF <- somascan2 |>
    subset(Target == "TNF-a")
somascanIL_6 <- somascan2 |>
    subset(Target == "IL-6")
somascanIL_1b <- somascan2 |>
    subset(Target == "IL-1b")

somascan_GSKB
somascan_GSKA
somascan_INS
somscan_TNF
somascanIL_6
somascanIL_1b

In [ ]:
# Finding GSK w/ Somascan 1
CNS <- somascan1[, colnames(somascan1) %in% c('RID','EXAMDATE','VISCODE2',
                                              'X24050.26','X3236.12','X3441.64', 'X4883.56', 
                                              'X5936.53', 'X5692.79', 'X2573.20', 'X4673.13', 'X3037.62')]
CNS <- CNS |>
    rename(GSK3A = X3441.64,
           GSK3B_1 = X24050.26,
           GSK3B_2 = X3236.12,
           Insulin = X4883.56,
           TNFa_1 = X5692.79,
           TNFa_2 = X5936.53,
           IL6_1 = X2573.20,
           IL6_2 = X4673.13,
           IL1b = X3037.62
          )
CNS

In [ ]:
# Checking for NA values
nrow(CNS)
cat("GSK3B - 1")
sum(is.na(CNS$GSK3B_1))

cat("GSK3B - 2")
sum(is.na(CNS$GSK3B_2))

cat("GSK3A")
sum(is.na(CNS$GSK3A))

cat("Insulin")
sum(is.na(CNS$Insulin))

cat("TNFA - 1")
sum(is.na(CNS$TNFa_1))

cat("TNFA - 2")
sum(is.na(CNS$TNFa_2))

cat("IL6 - 1")
sum(is.na(CNS$IL6_1))

cat("IL6 - 2")
sum(is.na(CNS$IL6_2))

cat("IL1b")
sum(is.na(CNS$IL1b))

In [ ]:
# Removing NA's
CNS1 <- CNS |>
  drop_na(GSK3B_1, GSK3B_2, GSK3A, Insulin, TNFa_1, TNFa_2, IL6_1, IL6_2, IL1b)

In [ ]:
nrow(CNS1)
table(CNS1$VISCODE2)

In [ ]:
# Taking only baseline measurements
CNS2 <- CNS1 |>
    subset(VISCODE2 == "bl")
nrow(CNS2)

In [ ]:
# Reading in Cohort
tbl1 <- read.csv("051_ADNI Merge Cohort V3")

In [ ]:
# Merging GSK2 w/ ADNI Merge Cohort
tbl2 <- merge(tbl1, CNS2, by = "RID")
tbl2 <- tbl2 |>
    rename(CNS_DATE = EXAMDATE.y,
           EXAMDATE = EXAMDATE.x,
           CNS_VISCODE = VISCODE2)
nrow(tbl2)

In [ ]:
# Scaling CNS Insulin resistance markers values
tbl2$GSK3B.1_z <- scale(tbl2$GSK3B_1)
tbl2$GSK3B.2_z <- scale(tbl2$GSK3B_2)
tbl2$GSK3A_z <- scale(tbl2$GSK3A)
tbl2$INS_z <- scale(tbl2$Insulin)
tbl2$TNFa.1_z <- scale(tbl2$TNFa_1)
tbl2$TNFa.2_z <- scale(tbl2$TNFa_2)
tbl2$IL6.1_z <- scale(tbl2$IL6_1)
tbl2$IL6.2_z <- scale(tbl2$IL6_2)
tbl2$IL1b_z <- scale(tbl2$IL1b)

# Preliminary Modelling to Test Which Is Potential CNS Marker

## ABETA

In [ ]:
# GSK3B - 1 
A_GSK3B.1 <- lm(ABETA_capped ~ GSK3B.1_z, data = tbl2)
summary(A_GSK3B.1)
    # Increase in GSKB leads to decreased ABETA (increased progression) 

In [ ]:
# GSK3B - 2 
A_GSK3B.2 <- lm(ABETA_capped ~ GSK3B.2_z, data = tbl2)
summary(A_GSK3B.2)
    # Increase in GSKB leads to decreased ABETA (increased progression)

In [ ]:
# GSK3A  
A_GSK3A <- lm(ABETA_capped ~ GSK3A_z, data = tbl2)
summary(A_GSK3A)
    # Increase in GSKA leads to decreased ABETA (increased progression)

In [ ]:
# Insulin
A_INSULIN <- lm(ABETA_capped ~ INS_z, data = tbl2)
summary(A_INSULIN)
    # Increase insulin leads to increased ABETA (decreased progression)

In [ ]:
# TNFa - 1
A_TNFa.1 <- lm(ABETA_capped ~ TNFa.1_z, data = tbl2)
summary(A_TNFa.1)
    # Increased TNFa_1 leads to decreased ABETA (increased progression)

In [ ]:
# TNFa - 2
A_TNFa.2 <- lm(ABETA_capped ~ TNFa.2_z, data = tbl2)
summary(A_TNFa.2)
    # Increased TNFa_2 leads to decreased ABETA (increased progression)

In [ ]:
# IL6 - 1
A_IL6.1 <- lm(ABETA_capped ~ IL6.1_z, data = tbl2)
summary(A_IL6.1)
    # Increased IL6_1 leads to decreased ABETA (increased progression)

In [ ]:
# IL6 - 2
A_IL6.2 <- lm(ABETA_capped ~ IL6.2_z, data = tbl2)
summary(A_IL6.2)

In [ ]:
# IL1b
A_IL1b <- lm(ABETA_capped ~ IL1b_z, data = tbl2)
summary(A_IL1b)

## TAU

In [ ]:
# GSK3B - 1 
T_GSK3B.1 <- lm(TAU_capped ~ GSK3B.1_z, data = tbl2)
summary(T_GSK3B.1)
    # Increase in GSKB leads to decreased TAU (decreased progression) 

In [ ]:
# GSK3B - 2
T_GSK3B.2 <- lm(TAU_capped ~ GSK3B.2_z, data = tbl2)
summary(T_GSK3B.2)
    # Increase in GSKB leads to decreased TAU (decreased progression) 

In [ ]:
# GSK3A
T_GSK3A <- lm(TAU_capped ~ GSK3A_z, data = tbl2)
summary(T_GSK3A)
    # Increase in GSKA leads to decreased TAU (decreased progression) 

In [ ]:
# Insulin
T_INSULIN <- lm(TAU_capped ~ INS_z, data = tbl2)
summary(T_INSULIN)
    # Non signficant

In [ ]:
# TNFa - 1
T_TNFa.1 <- lm(TAU_capped ~ TNFa.1_z, data = tbl2)
summary(T_TNFa.1)
    # Non signficant

In [ ]:
# TNFa - 2
T_TNFa.2 <- lm(TAU_capped ~ TNFa.2_z, data = tbl2)
summary(T_TNFa.2)
    # Increased TNFa causes decreased tau (decreased progession)

In [ ]:
# IL6 - 1
T_IL6.1 <- lm(TAU_capped ~ IL6.1_z, data = tbl2)
summary(T_IL6.1)
    # Increased IL6 - 1 causes decreased tau (decreased progession)

In [ ]:
# IL6 - 2
T_IL6.2 <- lm(TAU_capped ~ IL6.2_z, data = tbl2)
summary(T_IL6.2)
    # Increased IL6 - 2 causes decreased tau (decreased progession)

In [ ]:
# IL1b
T_IL1b <- lm(TAU_capped ~ IL1b_z, data = tbl2)
summary(T_IL1b)
    # Increased IL1b causes decreased tau (decreased progession)

## PTAU

In [ ]:
# GSK3B - 1 
P_GSK3B.1 <- lm(PTAU_capped ~ GSK3B.1_z, data = tbl2)
summary(P_GSK3B.1)
    # Increase in GSKB leads to decreased PTAU (decreased progression) 

In [ ]:
# GSK3B - 2
P_GSK3B.2 <- lm(PTAU_capped ~ GSK3B.2_z, data = tbl2)
summary(P_GSK3B.2)
    # Increase in GSKB leads to decreased PTAU (decreased progression) 

In [ ]:
# GSK3A
P_GSK3A <- lm(PTAU_capped ~ GSK3A_z, data = tbl2)
summary(P_GSK3A)
    # Increase in GSKA leads to decreased PTAU (decreased progression) 

In [ ]:
# INSULIN
P_INSULIN <- lm(PTAU_capped ~ INS_z, data = tbl2)
summary(P_INSULIN)
    # Non significant

In [ ]:
# TNFa - 1
P_TNFa.1 <- lm(PTAU_capped ~ TNFa.1_z, data = tbl2)
summary(P_TNFa.1)
    # Non significant

In [ ]:
# TNFa - 1
P_TNFa.2 <- lm(PTAU_capped ~ TNFa.2_z, data = tbl2)
summary(P_TNFa.2)
    #decreased progression

In [ ]:
# IL6 - 1
P_IL6.1 <- lm(PTAU_capped ~ IL6.1_z, data = tbl2)
summary(P_IL6.1)
    # decreased progression

In [ ]:
# IL6 - 2
P_IL6.2 <- lm(PTAU_capped ~ IL6.2_z, data = tbl2)
summary(P_IL6.2)
    # decreased progression

In [ ]:
# IL1b
P_IL1b <- lm(PTAU_capped ~ IL1b_z, data = tbl2)
summary(P_IL1b)
    # decreased progression

# Adjusted Linear Mixed Models

In [ ]:
# Got error code below in initial analysis so limiting it to only participants w/ 2+ data points
    #Error: number of observations (=843) <= number of random effects (=878) for term (VISCODEnum_capped | RID); the random-effects parameters and the residual variance (or scale parameter) are probably unidentifiable

In [ ]:
aggregate1 <- aggregate(VISCODE ~ RID, tbl2, length)
nrow(aggregate1) # 439 Participants w/ CNS biomarker data
aggregate2 <- aggregate1 |>
    subset(VISCODE >= 2)
nrow(aggregate2) # 257 w/ 2+ viscodes

# Keeping only participants with multiple datapoints
tbl3 <- merge(tbl2, aggregate2, by = "RID")

## Insulin

In [ ]:
tbl3 <- tbl3 |>
    # Dividing Insulin index into tertiles
    transform(INS_tertiles = cut(Insulin, breaks = quantile(Insulin, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE))

### ABETA

In [ ]:
# ABETA w/ raw insulin
INSULIN_A <- lmer(ABETA_capped ~ VISCODEnum_z * INS_z + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(INSULIN_A)

In [ ]:
# ABETA w/ INS Tertiles
INSULINtertiles_A <- lmer(ABETA_capped ~ VISCODEnum_z * INS_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(INSULINtertiles_A)

### TAU

In [ ]:
# TAU w/ raw insulin
INSULIN_T <- lmer(TAU_capped ~ VISCODEnum_z * INS_z + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(INSULIN_T)

In [ ]:
# TAU w/ raw insulin
INSULINtertiles_T <- lmer(TAU_capped ~ VISCODEnum_z * INS_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(INSULINtertiles_T)

### PTAU

In [ ]:
# PTAU w/ raw insulin
INSULIN_P <- lmer(PTAU_capped ~ VISCODEnum_z * INS_z + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(INSULIN_P)

In [ ]:
# TAU w/ raw insulin
INSULINtertiles_P <- lmer(PTAU_capped ~ VISCODEnum_z * INS_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(INSULINtertiles_P)

## GSK3A

In [ ]:
tbl3 <- tbl3 |>
    # Dividing GSK A index into tertiles
    transform(GSK3A_tertiles = cut(GSK3A, breaks = quantile(GSK3A, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) 

### ABETA

In [ ]:
# ABETA raw
GSK3A_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3A_z + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3A_A)

In [ ]:
# ABETA w/ Tertiles
GSK3Atertiles_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3A_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3Atertiles_A)

## GSK3B - 1 

In [ ]:
tbl3 <- tbl3 |>
    # Dividing GSK 3B - 1 index into tertiles
    transform(GSK3B.1_tertiles = cut(GSK3B_1, breaks = quantile(GSK3B_1, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) 

### ABETA

In [ ]:
# ABETA raw
GSK3B.1_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3B.1_z + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3B.1_A)

In [ ]:
# ABETA w/ tertiles
GSK3B.1tertiles_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3B.1_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3B.1tertiles_A)

## GSK3B - 2

In [ ]:
tbl3 <- tbl3 |>
    # Dividing GSK 3B - 2 index into tertiles
    transform(GSK3B.2_tertiles = cut(GSK3B_2, breaks = quantile(GSK3B_2, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) 

### ABETA

In [ ]:
# raw
GSK3B.2_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3B.2_z + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3B.2_A)

In [ ]:
# raw
GSK3B.2tertiles_A <- lmer(ABETA_capped ~ VISCODEnum_z * GSK3B.2_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(GSK3B.2tertiles_A)

## IL6 - 1

In [ ]:
tbl3 <- tbl3 |>
    # Dividing IL6 - 1 index into tertiles
    transform(IL6.1_tertiles = cut(IL6_1, breaks = quantile(IL6_1, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) 

In [ ]:
# IL6 - 1
IL6.1_tertiles_A <- lmer(ABETA_capped ~ VISCODEnum_z * IL6.1_tertiles + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(IL6.1_tertiles_A)

In [ ]:
# IL6 - 1
IL6.1_A <- lmer(ABETA_capped ~ VISCODEnum_z * IL6.1_z + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(IL6.1_A)

In [ ]:
# IL6 - 1
IL6.1_T <- lmer(TAU_capped ~ VISCODEnum_z * IL6.1_z + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(IL6.1_T)

In [ ]:
# IL6 - 1
IL6.1_P <- lmer(PTAU_capped ~ VISCODEnum_z * IL6.1_z + AGE_z + PTGENDER + PTEDUCAT_z + BMI_capped + APOE4 + (VISCODEnum_z | RID), data = tbl3)
summary(IL6.1_P)

# Other

In [ ]:
# Modelling relationship between GSK-B and TAU Pathology (linear model)
lm1 <- lm(TAU ~ GSKB1_z, data = tbl3)
summary(lm1)

In [ ]:
# GSK Beta x TAU
plm1 <- ggplot(tbl3, aes(GSKB1_z, TAU, colour = RID)) +
  geom_point(alpha = 0.5) +
  geom_abline(
    intercept = coef(lm1)[1],
    slope = coef(lm1)[2],
    linewidth = 1.1,
    colour = "black"
  ) +
  guides(colour = "none") +
  labs(
    title = "Ordinary linear model",
    subtitle = "One global line for all observations",
    x = "GSK",
    y = "TAU"
  ) +
  theme_minimal(base_size = 13)

plm1

### GSK-B (1) w/ ABETA

In [ ]:
# Modelling relationship between GSK-B and ABETA Pathology (linear model)
lm2 <- lm(ABETA_clean ~ GSKB1_z, data = tbl3)
summary(lm2)

In [ ]:
# GSK Beta x ABETA
plm2 <- ggplot(tbl3, aes(GSKB1_z, ABETA_clean, colour = RID)) +
  geom_point(alpha = 0.5) +
  geom_abline(
    intercept = coef(lm2)[1],
    slope = coef(lm2)[2],
    linewidth = 1.1,
    colour = "black"
  ) +
  guides(colour = "none") +
  labs(
    title = "Ordinary linear model",
    subtitle = "One global line for all observations",
    x = "GSK",
    y = "ABETA"
  ) +
  theme_minimal(base_size = 13)

plm2

### GSK-B (1) w/ PTAU

In [ ]:
# Modelling relationship between GSK-B and PTAU Pathology (linear model)
lm3 <- lm(PTAU_clean ~ GSKB1_z, data = tbl3)
summary(lm3)

In [ ]:
# GSK Beta x PTAU
plm3 <- ggplot(tbl3, aes(GSKB1_z, PTAU_clean, colour = RID)) +
  geom_point(alpha = 0.5) +
  geom_abline(
    intercept = coef(lm3)[1],
    slope = coef(lm3)[2],
    linewidth = 1.1,
    colour = "black"
  ) +
  guides(colour = "none") +
  labs(
    title = "Ordinary linear model",
    subtitle = "One global line for all observations",
    x = "GSK",
    y = "PTAU"
  ) +
  theme_minimal(base_size = 13)

plm3

### GSK Alpha & Tau

In [ ]:
lm4 <- lm(TAU ~ GSKA_z, data = tbl3)
summary(lm4)

In [ ]:
# GSK ALPHA x TAU
plm4 <- ggplot(tbl3, aes(GSKA_z, TAU, colour = RID)) +
  geom_point(alpha = 0.5) +
  geom_abline(
    intercept = coef(lm4)[1],
    slope = coef(lm4)[2],
    linewidth = 1.1,
    colour = "black"
  ) +
  guides(colour = "none") +
  labs(
    title = "Ordinary linear model",
    subtitle = "One global line for all observations",
    x = "GSK-A",
    y = "TAU"
  ) +
  theme_minimal(base_size = 13)

plm4

### GSK Alpha & ABETA

In [ ]:
lm5 <- lm(ABETA_clean ~ GSKA_z, data = tbl3)
summary(lm5)

In [ ]:
# GSK ALPHA x ABETA
plm5 <- ggplot(tbl3, aes(GSKA_z, ABETA_clean, colour = RID)) +
  geom_point(alpha = 0.5) +
  geom_abline(
    intercept = coef(lm5)[1],
    slope = coef(lm5)[2],
    linewidth = 1.1,
    colour = "black"
  ) +
  guides(colour = "none") +
  labs(
    title = "Ordinary linear model",
    subtitle = "One global line for all observations",
    x = "GSK-A",
    y = "ABETA"
  ) +
  theme_minimal(base_size = 13)

plm5

### GSK Alpha & PTAU

In [ ]:
lm6 <- lm(PTAU_clean ~ GSKA_z, data = tbl3)
summary(lm6)

In [ ]:
# GSK ALPHA x PTAU
plm6 <- ggplot(tbl3, aes(GSKA_z, PTAU_clean, colour = RID)) +
  geom_point(alpha = 0.5) +
  geom_abline(
    intercept = coef(lm6)[1],
    slope = coef(lm6)[2],
    linewidth = 1.1,
    colour = "black"
  ) +
  guides(colour = "none") +
  labs(
    title = "Ordinary linear model",
    subtitle = "One global line for all observations",
    x = "GSK-A",
    y = "PTAU"
  ) +
  theme_minimal(base_size = 13)

plm6

### GSK Beta (2) & TAU

In [ ]:
lm7 <- lm(TAU ~ GSKB2_z, data = tbl3)
summary(lm7)

In [ ]:
# GSK BETA 2 x PTAU
plm7 <- ggplot(tbl3, aes(GSKB2_z, TAU, colour = RID)) +
  geom_point(alpha = 0.5) +
  geom_abline(
    intercept = coef(lm7)[1],
    slope = coef(lm7)[2],
    linewidth = 1.1,
    colour = "black"
  ) +
  guides(colour = "none") +
  labs(
    title = "Ordinary linear model",
    subtitle = "One global line for all observations",
    x = "GSKB (2)",
    y = "TAU"
  ) +
  theme_minimal(base_size = 13)

plm7

### GSK Beta (2) & ABETA

In [ ]:
lm8 <- lm(ABETA_clean ~ GSKB2_z, data = tbl3)
summary(lm8)

In [ ]:
# GSK BETA 2 x ABETA
plm8 <- ggplot(tbl3, aes(GSKB2_z, ABETA_clean, colour = RID)) +
  geom_point(alpha = 0.5) +
  geom_abline(
    intercept = coef(lm8)[1],
    slope = coef(lm8)[2],
    linewidth = 1.1,
    colour = "black"
  ) +
  guides(colour = "none") +
  labs(
    title = "Ordinary linear model",
    subtitle = "One global line for all observations",
    x = "GSKB (2)",
    y = "ABETA"
  ) +
  theme_minimal(base_size = 13)

plm8

### GSK Beta (2) & PTAU

In [ ]:
lm9 <- lm(PTAU_clean ~ GSKB2_z, data = tbl3)
summary(lm9)

In [ ]:
# GSK BETA 2 x PTAU
plm9 <- ggplot(tbl3, aes(GSKB2_z, PTAU_clean, colour = RID)) +
  geom_point(alpha = 0.5) +
  geom_abline(
    intercept = coef(lm9)[1],
    slope = coef(lm9)[2],
    linewidth = 1.1,
    colour = "black"
  ) +
  guides(colour = "none") +
  labs(
    title = "Ordinary linear model",
    subtitle = "One global line for all observations",
    x = "GSKB (2)",
    y = "PTAU"
  ) +
  theme_minimal(base_size = 13)

plm9

In [ ]:
aggregate1 <- aggregate(VISCODE ~ RID, tbl3, length)
nrow(aggregate1) # 478 Participants w/ GSK data @ bl
aggregate2 <- aggregate1 |>
    subset(VISCODE >= 2)
nrow(aggregate2) # 283 w/ 2+ viscodes

# Keeping only participants with multiple datapoints
tbl4 <- merge(tbl3, aggregate2, by = "RID")

In [ ]:
# Dividing participants into GSK tertiles
tbl5 <- tbl4 |>
    # Dividing GSK B1 into tertiles
    transform(GSKB1_tertiles = cut(GSK_3_Beta1, breaks = quantile(GSK_3_Beta1, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) |>
    # Dividing GSK B2 into tertiles
    transform(GSKB2_tertiles = cut(GSK_3_Beta2, breaks = quantile(GSK_3_Beta2, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) |>
    # Dividing GSK A index into tertiles
    transform(GSKA_tertiles = cut(GSK_3_Alpha, breaks = quantile(GSK_3_Alpha, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) |>
    # Dividing GSK A index into tertiles
    transform(INS_tertiles = cut(Insulin, breaks = quantile(Insulin, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) 

In [ ]:
# LMM for ABETA & Insulin
ins_ab <- lmer(ABETA_clean ~ VISCODEnum_z * INS_tertiles + (VISCODEnum_z | RID), data = tbl5)
summary(ins_ab)
    # Reveals that increased CSF insulin is correlated to less abeta

### GSK-3B (1) & P-Tau

In [ ]:
# LMM for PTAU & GSKB1_z
m1 <- lmer(PTAU_clean ~ VISCODEnum_z * GSKB1_tertiles + (VISCODEnum_z | RID), data = tbl5)
summary(m1)
    # Reveals that increased GSKB1_z decreases ptau

In [ ]:
tbl5 <- tbl5 |>
    mutate(pred_m1 = predict(m1))
m1_g <- ggpredict(m1, terms = c("VISCODEnum_z", "GSKB1_tertiles")) |>
    plot() +
    labs(
        title = "Predicted p-Tau Over Time Stratified by GSK-3B (1) Tertile",
        x = "Months Since Baseline (scaled)",
        y = "Predicted p-Tau (pg/mL)",
        color = "GSK-3B Tertile"
      ) +
  theme_minimal(base_size = 13)
m1_g

### GSK-3A & AB

In [ ]:
# LMM for ABETA & GSKA_z
m2 <- lmer(ABETA_clean ~ VISCODEnum_z * GSKA_tertiles + (VISCODEnum_z | RID), data = tbl5)
summary(m2)
    # Reveals that increased GSKA is associated with drop in ABETA (linked to increased AD)

In [ ]:
tbl5 <- tbl5 |>
    mutate(pred_m2 = predict(m2))
m2_g <- ggpredict(m2, terms = c("VISCODEnum_z", "GSKA_tertiles")) |>
    plot() +
    labs(
        title = "Predicted AB Over Time Stratified by GSK-3A Tertile",
        x = "Months Since Baseline (scaled)",
        y = "Predicted AB (pg/mL)",
        color = "GSK-3A Tertile"
      ) +
  theme_minimal(base_size = 13)
m2_g

In [ ]:
# LMM for ABETA & GSKA_z (ADJUSTED!)
m3 <- lmer(ABETA_clean ~ VISCODEnum_z * GSKA_tertiles + BMI_z + PTGENDER + AGE_z + APOE4 + DIAGNOSIS_bl + PTEDUCAT_z + (VISCODEnum_z | RID), data = tbl5)
summary(m3)
    # Reveals that increased GSKA is associated with drop in ABETA (linked to increased AD)

In [ ]:
tbl5 <- tbl5 |>
    mutate(pred_m3 = predict(m3))
m3_g <- ggpredict(m3, terms = c("VISCODEnum_z", "GSKA_tertiles")) |>
    plot() +
    labs(
        title = "Predicted AB Over Time Stratified by GSK-3A Tertile",
        x = "Months Since Baseline (scaled)",
        y = "Predicted AB (pg/mL)",
        color = "GSK-3A Tertile"
      ) +
  theme_minimal(base_size = 13)
m3_g

In [ ]:
# Base model - scaled & refit with Bobyqa Optimizer
m1_a <- lmer(
    ABETA_clean ~ VISCODEnum_z + DIAGNOSIS_bl + (VISCODEnum_z | RID),
    data = tbl5,
    control = lmerControl(optimizer = "bobyqa"))
summary(m1_a)

In [ ]:
# LMM for ABETA & GSKB1 Tertiles (ADJUSTED!)
m4 <- lmer(ABETA_clean ~ VISCODEnum_z * GSKB1_tertiles + BMI_z + PTGENDER + AGE_z + APOE4 + DIAGNOSIS_bl + PTEDUCAT_z + (VISCODEnum_z | RID), data = tbl5)
summary(m4)
    # Reveals that increased GSKA is associated with drop in ABETA (linked to increased AD)

In [ ]:
tbl5 <- tbl5 |>
    mutate(pred_m4 = predict(m4))
m4_g <- ggpredict(m4, terms = c("VISCODEnum_z", "GSKB1_tertiles")) |>
    plot() +
    labs(
        title = "Predicted AB Over Time Stratified by GSK-3B (1) Tertile",
        x = "Months Since Baseline (scaled)",
        y = "Predicted AB (pg/mL)",
        color = "GSK-3B (1) Tertile"
      ) +
  theme_minimal(base_size = 13)
m4_g

In [ ]:
# LMM for ABETA & GSKB2 Tertiles (ADJUSTED!)
m5 <- lmer(ABETA_clean ~ VISCODEnum_z * GSKB2_tertiles + BMI_z + PTGENDER + AGE_z + APOE4 + DIAGNOSIS_bl + PTEDUCAT_z + (VISCODEnum_z | RID), data = tbl5)
summary(m5)
    # Reveals that increased GSKA is associated with drop in ABETA (linked to increased AD)